In [ ]:
from pathlib import Path
import pandas as pd
idx_slice = pd.IndexSlice
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)
import plotly.graph_objects as go
import plotly.express as px

import sys
sys.path.append('..')
from scripts.plot_helpers import (
    chdir_to_root_dir,
    read_stats_dict,
    read_csv_nafix,
    prepare_dataframe,
    nice_title,
    load_plot_config,
    get_scen_col_function,
    save_plotly_fig,
    apply_standard_styling,
    update_layout,
    h2o_cost_bar_fig)

chdir_to_root_dir()


In [ ]:
yaml_config = load_plot_config()

RUN_NAME_PREFIX = yaml_config['run']['name_prefix']
SUMMARY_VERSION = yaml_config['run']['summary_version']
SDIR = Path.cwd() / "results" / f"{RUN_NAME_PREFIX}_summary_{SUMMARY_VERSION}"
SDIR.mkdir(exist_ok=True, parents=True)

COUNTRIES = yaml_config['data']['countries']
YEARS = yaml_config['data']['years']
INDEX_LEVELS_TO_DROP = yaml_config['data']['index_levels_to_drop']

IDX_GROUP = idx_slice[[RUN_NAME_PREFIX], :, COUNTRIES, YEARS]
IDX_GROUP_NAME = "".join(COUNTRIES) + "_MAIN"

print(f"Index Group Name: {IDX_GROUP_NAME}")
print(f"Index Group: {IDX_GROUP}")

# Scenario filtering
SCEN_FILTER = yaml_config['data']['scen_filter']
scen_order = yaml_config['data']['scen_order']

# Get scenario column function
scen_col_func_name = yaml_config['data']['scen_col_function']
set_scen_col = get_scen_col_function(scen_col_func_name)
print(f"Using scenario column function: {scen_col_func_name}")

# Plot dimensions
width = yaml_config['plot']['width']
heights = yaml_config['plot']['heights']

# Default figure kwargs
fig_kwargs = yaml_config['plot']['default_kwargs'].copy()
fig_kwargs['width'] = width
fig_kwargs['height'] = heights['medium']

print(f"\nConfiguration loaded successfully!")
print(f"Countries: {COUNTRIES}")
print(f"Years: {YEARS}")

In [ ]:
# marginal_prices_prepared.csv is generated by make_stats_dicts.py.
# It contains all bus marginal prices in long-form (country, year, scen, variable, value).
marginal_prices = read_csv_nafix(SDIR / "marginal_prices_prepared.csv")
marginal_prices = marginal_prices.query("scen in @SCEN_FILTER") if SCEN_FILTER else marginal_prices

available_countries = marginal_prices["country"].unique().tolist()
missing_countries = [c for c in COUNTRIES if c not in available_countries]
if missing_countries:
    print(f"⚠️  Countries not found in prepared data: {missing_countries}")

print(f"Loaded {len(marginal_prices)} rows. Variables: {sorted(marginal_prices['variable'].unique().tolist())}")


In [ ]:
df = marginal_prices.query("variable in ['H2 export bus']").copy()


## Marginal prices | H2 export price

In [ ]:
# marginal_prices_prepared.csv is now generated at the end of make_stats_dicts.py.
# It includes all bus carriers (H2, H2O, etc.) — no need to re-save here.
print(f"marginal_prices_prepared.csv was saved by make_stats_dicts.py to:\n{SDIR / 'marginal_prices_prepared.csv'}")


In [ ]:
ISO2_TO_NAME = yaml_config["data"]["iso2_to_name"]
VOLUME_LABELS = {int(k): v for k, v in yaml_config["data"]["volume_labels"].items()}

def marginal_price_dumbbell_fig(df, year, data_start="low", data_mid="mid", data_end="high",
                                xaxis_title="Marginal price (€/MWh<sub>H2</sub>)",
                                conversion_factor=0.03333,
                                xaxis2_title="Marginal price (€/kg H2)",
                                xaxis2_color="#a8508c",
                                volume_labels=None):

    if volume_labels is None:
        volume_labels = VOLUME_LABELS

    df = df[(df.year == year)].copy()
    countries = df["country"].unique()

    # Build legend names with volume suffixes
    yr_volumes = volume_labels.get(year, {})
    label_start = f"{data_start} ({yr_volumes[data_start]})" if data_start in yr_volumes else data_start
    label_mid = f"{data_mid} ({yr_volumes[data_mid]})" if data_mid in yr_volumes else data_mid
    label_end = f"{data_end} ({yr_volumes[data_end]})" if data_end in yr_volumes else data_end

    vals_start, vals_mid, vals_end, valid = [], [], [], []
    skipped = []

    for country in countries:
        s = df.loc[(df.scen == f"{country}-{data_start}") & (df.country == country), "value"]
        m = df.loc[(df.scen == f"{country}-{data_mid}") & (df.country == country), "value"]
        e = df.loc[(df.scen == f"{country}-{data_end}") & (df.country == country), "value"]
        if len(s) > 0 and len(e) > 0:
            vals_start.append(s.values[0])
            vals_end.append(e.values[0])
            vals_mid.append(m.values[0] if len(m) > 0 else None)
            valid.append(country)
        else:
            missing = []
            if len(s) == 0:
                missing.append(data_start)
            if len(e) == 0:
                missing.append(data_end)
            skipped.append(f"{country} (missing: {', '.join(missing)})")

    # Sort descending by data_start value (highest at bottom, lowest at top)
    order = sorted(range(len(vals_start)), key=lambda i: vals_start[i], reverse=True)
    valid = [valid[i] for i in order]
    vals_start = [vals_start[i] for i in order]
    vals_end = [vals_end[i] for i in order]
    vals_mid = [vals_mid[i] for i in order]

    # Map ISO2 codes to full country names
    valid_names = [ISO2_TO_NAME.get(c, c) for c in valid]

    # Compute max value across all data points for axis range
    all_data_vals = vals_start + vals_end + [v for v in vals_mid if v is not None]
    max_val = max(all_data_vals) if all_data_vals else 1

    # Rebuild connecting lines from sorted data, spanning the full range including mid
    line_x, line_y = [], []
    for vs, vm, ve, name in zip(vals_start, vals_mid, vals_end, valid_names):
        all_vals = [v for v in [vs, vm, ve] if v is not None]
        line_x.extend([min(all_vals), max(all_vals), None])
        line_y.extend([name, name, None])

    fig = go.Figure(data=[
        go.Scatter(x=line_x, y=line_y, mode="lines", showlegend=False,
                   marker=dict(color="grey")),
        go.Scatter(
            x=vals_start, y=valid_names, mode="markers+text", name=label_start,
            text=[f"{v:.0f}" for v in vals_start], textposition="middle left",
            textfont=dict(size=11), marker=dict(color="#99bdcc", size=13),
        ),
        go.Scatter(
            x=[v for v in vals_mid if v is not None],
            y=[n for n, v in zip(valid_names, vals_mid) if v is not None],
            mode="markers", name=label_mid,
            marker=dict(color="#669db2", size=13),
        ),
        go.Scatter(
            x=vals_end, y=valid_names, mode="markers+text", name=label_end,
            text=[f"{v:.0f}" for v in vals_end], textposition="middle right",
            textfont=dict(size=11), marker=dict(color="#005b7f", size=13),
        ),
        # Invisible trace to force xaxis2 to render
        go.Scatter(
            x=[80 * conversion_factor, max_val * 1.1 * conversion_factor],
            y=[valid_names[0], valid_names[0]],
            xaxis="x2",
            mode="markers",
            marker=dict(opacity=0),
            showlegend=False,
            hoverinfo="skip",
        ),
    ])

    fig.update_layout(
        height=max(500, len(valid_names) * 45),
        width=680,
        legend=dict(x=0.8, y=1, xanchor="left", yanchor="top"),
        legend_itemclick=False,
        xaxis_title=xaxis_title,
        xaxis=dict(range=[80, max_val * 1.1]),
        yaxis=dict(title="", categoryorder="array", categoryarray=valid_names),
        # Secondary x-axis (top) showing converted values
        xaxis2=dict(
            title=dict(text=xaxis2_title, font=dict(color=xaxis2_color)),
            overlaying="x",
            side="top",
            range=[80 * conversion_factor, max_val * 1.1 * conversion_factor],
            tickfont=dict(color=xaxis2_color),
            linecolor=xaxis2_color,
            tickcolor=xaxis2_color,
        ),
    )
    return fig, skipped


In [ ]:
fig, skipped = marginal_price_dumbbell_fig(df, year=2050, data_start="low", data_end="high")
if skipped:
    print(f"Skipped: {', '.join(skipped)}")
fig.show()
save_plotly_fig(df, fig, Path(""), "marginal_price_dumbbell_fig_2050")

In [ ]:
fig, skipped = marginal_price_dumbbell_fig(df, year=2035, data_start="low", data_end="high")
if skipped:
    print(f"Skipped: {', '.join(skipped)}")
fig.show()
save_plotly_fig(df, fig, Path(""), "marginal_price_dumbbell_fig_2035")

## H2O Cost per H2 from Electrolysis – Component Breakdown

Average of all export scenarios (low / mid / high).  
**Method:** `(capex + opex)_component [bn €]  /  H2_Electrolysis [TWh]  × 1000  =  € / MWh_H2`

In [ ]:

# ── Index columns shared across all summary CSVs ─────────────────────────────
_IDX = ["run_name_prefix", "run_name", "country", "year",
        "simpl", "clusters", "ll", "opts", "sopts",
        "discountrate", "demand", "h2export"]

# H2O-related cost technologies (capex + opex summed per component)
H2O_COMPONENTS = [
    "desalination",
    "seawater",
    "H2O pipeline",
    "H2O store",
    "H2O store charger",
    "H2O store discharger",
    "H2O generator",
]

# ── Load raw tables ───────────────────────────────────────────────────────────
_capex = read_csv_nafix(SDIR / "costs_dict_capex.csv")
_opex  = read_csv_nafix(SDIR / "costs_dict_opex.csv")
_h2    = read_csv_nafix(SDIR / "balance_dict_H2.csv")

# ── Filter to configured countries & years ────────────────────────────────────
def _filt(df):
    return df[df["country"].isin(COUNTRIES) & df["year"].isin(YEARS)].copy()

capex_f = _filt(_capex)
opex_f  = _filt(_opex)
h2_f    = _filt(_h2)

# ── Determine which H2O components are actually present ──────────────────────
present_comps = [c for c in H2O_COMPONENTS
                 if c in capex_f.columns or c in opex_f.columns]
print(f"H2O cost components found: {present_comps}")

# ── Merge capex + opex on full index key ──────────────────────────────────────
cap_sub = (capex_f[_IDX + [c for c in present_comps if c in capex_f.columns]]
           .rename(columns={c: f"{c}__cap" for c in present_comps
                             if c in capex_f.columns}))
opx_sub = (opex_f[_IDX + [c for c in present_comps if c in opex_f.columns]]
           .rename(columns={c: f"{c}__opx" for c in present_comps
                             if c in opex_f.columns}))

combined = cap_sub.merge(opx_sub, on=_IDX, how="inner")

# Sum capex + opex per component [bn €]
h2o_costs = combined[_IDX].copy()
for comp in present_comps:
    cap_v = combined[f"{comp}__cap"].fillna(0) if f"{comp}__cap" in combined.columns \
            else pd.Series(0.0, index=combined.index)
    opx_v = combined[f"{comp}__opx"].fillna(0) if f"{comp}__opx" in combined.columns \
            else pd.Series(0.0, index=combined.index)
    h2o_costs[comp] = cap_v.values + opx_v.values

# ── H2 from electrolysis [TWh] ────────────────────────────────────────────────
h2_elec = h2_f[_IDX + ["H2 Electrolysis"]].copy()
h2_elec["H2 Electrolysis"] = h2_elec["H2 Electrolysis"].abs()

# ── Merge & compute € / MWh_H2 ───────────────────────────────────────────────
# bn € / TWh × 1000  →  € / MWh_H2
df_h2o = h2o_costs.merge(h2_elec, on=_IDX, how="inner")
for comp in present_comps:
    df_h2o[comp] = df_h2o[comp] / df_h2o["H2 Electrolysis"] * 1000

# ── Add scenario label via set_scen_col (same mapping as all other charts) ───
df_h2o = set_scen_col(df_h2o, index_levels_to_drop=INDEX_LEVELS_TO_DROP)

# ── Per-scenario data (no averaging) ─────────────────────────────────────────
df_scen = df_h2o[["country", "year", "scen"] + present_comps].copy()
df_scen["total"] = df_scen[present_comps].sum(axis=1)


# Drop components with negligible contribution

nonzero_comps = [c for c in present_comps if df_scen[c].abs().max() > 0.01]
print(f"Non-trivial components: {nonzero_comps}")
print()
print(df_scen[["country", "year", "scen", "total"] + nonzero_comps]
      .sort_values(["year", "scen"])
      .round(2)
      .to_string(index=False))


In [ ]:
df_h2o

In [ ]:

# ── Colour palette for H2O components (loaded from plot_summary.yaml) ───────
H2O_COMP_COLORS = yaml_config["charts"]["h2o_cost_breakdown"]["color_discrete_map"]

# ── Plots ─────────────────────────────────────────────────────────────────────
for _yr in YEARS:
    _fig = h2o_cost_bar_fig(df_scen, year=_yr, components=nonzero_comps,
                             scen_order_list=scen_order,
                             component_colors=H2O_COMP_COLORS)
    _fig.show()
    save_plotly_fig(df_scen, _fig, Path(""), f"h2o_cost_breakdown_{_yr}")
